# Módulo 7: Entrenar y Serializar Modelo para Despliegue

Este notebook entrena el modelo final de churn y lo guarda para que el servicio FastAPI lo pueda usar.

## Pasos
1. Entrenar modelo de churn
2. Serializar modelo + vectorizer
3. Verificar carga
4. Instrucciones para despliegue

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score

np.random.seed(42)

## 1. Crear y preparar datos

In [ ]:
# Dataset de churn (mismo que módulos 4-5)
n = 2000
np.random.seed(42)

df = pd.DataFrame({
    'contract': np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.5, 0.3, 0.2]),
    'tenure': np.random.randint(1, 72, n),
    'monthly_charges': np.random.uniform(20, 100, n).round(2),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'No'], n, p=[0.35, 0.45, 0.2]),
    'online_security': np.random.choice(['Yes', 'No', 'No internet'], n, p=[0.3, 0.5, 0.2]),
    'tech_support': np.random.choice(['Yes', 'No', 'No internet'], n, p=[0.3, 0.5, 0.2]),
    'payment_method': np.random.choice(
        ['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n
    ),
})
df['total_charges'] = (df['monthly_charges'] * df['tenure']).round(2)

prob_churn = (
    0.1
    + 0.3 * (df['contract'] == 'Month-to-month')
    - 0.15 * (df['contract'] == 'Two year')
    - 0.005 * df['tenure']
    + 0.003 * df['monthly_charges']
    + 0.1 * (df['internet_service'] == 'Fiber optic')
    - 0.1 * (df['online_security'] == 'Yes')
    + 0.05 * (df['payment_method'] == 'Electronic check')
).clip(0.05, 0.95)
df['churn'] = (np.random.random(n) < prob_churn).astype(int)

print(f"Dataset: {df.shape}, Churn rate: {df['churn'].mean():.2%}")

In [ ]:
# Split: entrenar con 80%, test con 20%
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

y_train = df_train.pop('churn').values
y_test = df_test.pop('churn').values

# Features para el modelo
features = ['contract', 'tenure', 'monthly_charges', 'total_charges',
            'internet_service', 'online_security', 'tech_support', 'payment_method']

print(f"Train: {len(df_train)}, Test: {len(df_test)}")
print(f"Features: {features}")

## 2. Entrenar modelo final

In [ ]:
# Vectorizar
train_dicts = df_train[features].to_dict(orient='records')
test_dicts = df_test[features].to_dict(orient='records')

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_test = dv.transform(test_dicts)

print(f"Features codificadas: {X_train.shape[1]}")

# Entrenar
modelo = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
modelo.fit(X_train, y_train)

# Evaluar
y_pred_proba = modelo.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, (y_pred_proba >= 0.5).astype(int))

print(f"\nResultados en TEST:")
print(f"  AUC:      {auc:.4f}")
print(f"  Accuracy: {acc:.4f}")

## 3. Serializar (guardar) modelo

In [ ]:
# Guardar en la carpeta del servicio
output_dir = '../servicio'

joblib.dump(modelo, f'{output_dir}/modelo_churn.joblib')
joblib.dump(dv, f'{output_dir}/vectorizer.joblib')

print(f"Modelo guardado en: {output_dir}/modelo_churn.joblib")
print(f"Vectorizer guardado en: {output_dir}/vectorizer.joblib")

## 4. Verificar que se carga correctamente

In [ ]:
# Simular lo que hace el servicio
modelo_cargado = joblib.load(f'{output_dir}/modelo_churn.joblib')
dv_cargado = joblib.load(f'{output_dir}/vectorizer.joblib')

# Predecir con un cliente de prueba
cliente_test = {
    'contract': 'Month-to-month',
    'tenure': 3,
    'monthly_charges': 75.0,
    'total_charges': 225.0,
    'internet_service': 'Fiber optic',
    'online_security': 'No',
    'tech_support': 'No',
    'payment_method': 'Electronic check',
}

X_cliente = dv_cargado.transform([cliente_test])
prob = modelo_cargado.predict_proba(X_cliente)[0, 1]

print(f"Cliente de prueba:")
for k, v in cliente_test.items():
    print(f"  {k}: {v}")
print(f"\nProbabilidad de churn: {prob:.4f}")
print(f"Decisión: {'CHURN' if prob >= 0.5 else 'NO CHURN'}")

## 5. Despliegue

Ahora que el modelo está guardado en `servicio/`, puedes:

### Opción A: Ejecutar localmente
```bash
cd servicio
uvicorn main:app --reload
# Abrir http://localhost:8000/docs
```

### Opción B: Con Docker
```bash
cd servicio
docker build -t servicio-churn .
docker run -p 8000:8000 servicio-churn
```

### Probar
```bash
python test_api.py
```